# Audio Fractal Lab — Pangea-Earth

**Same fractal toolkit applied to beats. Same owl, different coordinate.**

1. Rhythmic complexity dimension (box-counting on onset timing)
2. Melanin EQ (absorption spectrum as adaptive EQ curve)
3. Mycelium arrangement analysis (does the beat grow toward signal?)
4. Turing reaction-diffusion hi-hat generation
5. Sierpiński stem processing (cheap filters first)

Load beats from Drive. Analyze. Transform. Save.

Guinea Pig Trench LLC

In [ ]:
#@title 1. Setup — Mount Drive, find beats
import os, time, json
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

!pip install -q librosa soundfile pydub

import librosa
import soundfile as sf
import matplotlib.pyplot as plt

DRIVE_BASE = Path('/content/drive/MyDrive')
AUDIO_DIR = DRIVE_BASE / 'Guinea Pig Trench' / 'audio_research'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Find beats — actual locations on Drive
beat_dirs = [
    DRIVE_BASE / 'workspace' / 'audio_masters',
    DRIVE_BASE / 'MotoG_Backup' / 'Download',
    DRIVE_BASE / 'MotoG_Backup_2026-03-20' / 'Download',
    DRIVE_BASE / 'Guinea Pig Trench' / 'beats',
]

all_beats = []
for d in beat_dirs:
    if d.exists():
        for ext in ['*.wav', '*.mp3', '*.mp4']:
            all_beats.extend(d.glob(ext))

# Also check for stems
stem_dirs = [
    DRIVE_BASE / 'workspace' / 'stems',
    DRIVE_BASE / 'Guinea Pig Trench' / 'stems',
]

all_stems = []
for d in stem_dirs:
    if d.exists():
        all_stems.extend(d.rglob('*.wav'))

SESSION_START = time.time()
SESSION_LIMIT = 80 * 60
def time_left(): return max(0, SESSION_LIMIT - (time.time() - SESSION_START))

print(f'Beats found: {len(all_beats)}')
print(f'Stems found: {len(all_stems)}')
for b in all_beats[:15]:
    print(f'  {b.parent.name}/{b.name}')
if len(all_beats) > 15:
    print(f'  ... and {len(all_beats)-15} more')
print(f'\nSession budget: {SESSION_LIMIT//60} min')

In [ ]:
#@title 2. Rhythmic Complexity Dimension — box-counting on onset timing

def rhythmic_dimension(y, sr, hop=512):
    """Measure fractal dimension of rhythmic complexity.
    High D = syncopated, complex (Dilla, Heatmakerz).
    Low D = simple, repetitive (basic loops).
    Same box-counting algorithm as the sieve."""
    
    # Get onset strength envelope
    oenv = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    
    # Detect onsets
    onsets = librosa.onset.onset_detect(y=y, sr=sr, hop_length=hop, units='time')
    if len(onsets) < 4:
        return 0, []
    
    # Box-counting across time scales
    duration = len(y) / sr
    scales = []
    counts = []
    
    for div in [2, 4, 8, 16, 32, 64, 128, 256]:
        box_size = duration / div
        if box_size <= 0:
            continue
        boxes = set()
        for t in onsets:
            boxes.add(int(t / box_size))
        if len(boxes) > 0:
            scales.append(1.0 / box_size)
            counts.append(len(boxes))
    
    if len(scales) < 3:
        return 0, []
    
    log_s = np.log(scales)
    log_c = np.log(counts)
    D = np.polyfit(log_s, log_c, 1)[0]
    
    return D, onsets


# Analyze available beats
print('=== Rhythmic Complexity Analysis ===')
print('D > 1.5 = complex/syncopated (Dilla, trap fills)')
print('D ~ 1.0 = moderate (standard boom-bap)')
print('D < 0.8 = simple/repetitive (basic loop)')
print()

beat_dimensions = []
beats_to_analyze = all_beats[:20]  # First 20

for beat_path in beats_to_analyze:
    try:
        y, sr = librosa.load(str(beat_path), sr=22050, duration=60)
        D, onsets = rhythmic_dimension(y, sr)
        if D > 0:
            tempo = librosa.beat.tempo(y=y, sr=sr)[0]
            beat_dimensions.append({
                'name': beat_path.name,
                'D': round(D, 3),
                'tempo': round(float(tempo), 1),
                'onsets': len(onsets),
            })
            complexity = 'complex' if D > 1.5 else 'moderate' if D > 0.8 else 'simple'
            print(f'  {beat_path.name}: D={D:.3f} ({complexity}), {tempo:.0f} BPM, {len(onsets)} onsets')
    except Exception as e:
        print(f'  {beat_path.name}: error — {e}')

if beat_dimensions:
    # Sort by complexity
    beat_dimensions.sort(key=lambda x: x['D'], reverse=True)
    print(f'\nMost complex: {beat_dimensions[0]["name"]} (D={beat_dimensions[0]["D"]})')
    print(f'Simplest: {beat_dimensions[-1]["name"]} (D={beat_dimensions[-1]["D"]})')
    
    # Save
    (AUDIO_DIR / 'beat_dimensions.json').write_text(json.dumps(beat_dimensions, indent=2))
    print(f'\nSaved to {AUDIO_DIR / "beat_dimensions.json"}')
else:
    print('No beats analyzed — check Drive paths above.')

In [ ]:
#@title 3. Melanin EQ — adaptive absorption curve

def melanin_eq(y, sr, mode='eumelanin'):
    """Apply melanin-inspired adaptive EQ.
    Eumelanin = warm, dark, absorbs highs (brown/black skin).
    Pheomelanin = brighter, mid-presence shoulder (red/warm tones).
    Blend between them like the skin tone spectrum."""
    import scipy.signal as signal
    
    # Measure track brightness
    S = np.abs(librosa.stft(y))
    freqs = librosa.fft_frequencies(sr=sr)
    mag = np.mean(S, axis=1)
    spectral_centroid = np.sum(freqs * mag) / (np.sum(mag) + 1e-8)
    brightness = np.clip(spectral_centroid / 4000, 0.3, 2.0)
    
    print(f'  Spectral centroid: {spectral_centroid:.0f} Hz (brightness: {brightness:.2f})')
    
    if mode == 'eumelanin':
        # Dark, warm — gentle high rolloff, sub warmth
        # Low shelf boost at 80Hz
        b_low, a_low = signal.iirfilter(2, 80, btype='low', fs=sr, output='ba', ftype='butter')
        y_low = signal.lfilter(b_low, a_low, y) * 0.15
        
        # High shelf cut — melanin absorbs UV (high freq)
        b_high, a_high = signal.iirfilter(2, 8000, btype='high', fs=sr, output='ba', ftype='butter')
        y_high = signal.lfilter(b_high, a_high, y) * (-0.2 / brightness)
        
        result = y + y_low + y_high
        
    elif mode == 'pheomelanin':
        # Brighter, mid-presence shoulder around 2-5kHz
        b_mid, a_mid = signal.iirfilter(2, [2000, 5000], btype='band', fs=sr, output='ba', ftype='butter')
        y_mid = signal.lfilter(b_mid, a_mid, y) * (0.3 * brightness)
        
        result = y + y_mid
        
    else:
        # Blend — ratio between eu and pheo
        ratio = float(mode)  # 0.0 = full pheomelanin, 1.0 = full eumelanin
        eu = melanin_eq(y, sr, 'eumelanin')
        ph = melanin_eq(y, sr, 'pheomelanin')
        result = eu * ratio + ph * (1 - ratio)
    
    # Normalize
    peak = np.max(np.abs(result))
    if peak > 0:
        result = result * (np.max(np.abs(y)) / peak)
    
    return result


# Demo on first beat
if all_beats:
    demo_beat = all_beats[0]
    print(f'=== Melanin EQ Demo: {demo_beat.name} ===')
    y, sr = librosa.load(str(demo_beat), sr=22050, duration=30)
    
    print('\nEumelanin (warm, dark):')
    y_eu = melanin_eq(y, sr, 'eumelanin')
    
    print('\nPheomelanin (bright, present):')
    y_ph = melanin_eq(y, sr, 'pheomelanin')
    
    # Save both versions
    sf.write(str(AUDIO_DIR / f'{demo_beat.stem}_eumelanin.wav'), y_eu, sr)
    sf.write(str(AUDIO_DIR / f'{demo_beat.stem}_pheomelanin.wav'), y_ph, sr)
    
    # Plot comparison
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    for i, (label, audio) in enumerate([('Original', y), ('Eumelanin', y_eu), ('Pheomelanin', y_ph)]):
        S = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
        librosa.display.specshow(S, sr=sr, ax=axes[i], x_axis='time', y_axis='hz')
        axes[i].set_title(label)
        axes[i].set_ylim(0, 10000)
    plt.tight_layout()
    plt.savefig(str(AUDIO_DIR / 'melanin_eq_comparison.png'), dpi=150)
    plt.show()
    print(f'\nSaved to {AUDIO_DIR}')
else:
    print('No beats found on Drive.')

In [ ]:
#@title 4. Turing Hi-Hat Generator — reaction-diffusion percussion
from scipy.ndimage import convolve1d

def turing_hihat_pattern(sr, duration_sec, tempo, f=0.055, k=0.062, seed=42):
    """Generate organic hi-hat pattern via 1D reaction-diffusion.
    Each beat gets unique, non-repetitive percussion.
    Same math as the melanin Turing patterns — spots/stripes in time."""
    
    beat_dur = 60.0 / tempo
    grid_size = int((duration_sec / beat_dur) * 16)  # 16th-note grid
    
    rng = np.random.RandomState(seed)
    u = np.ones(grid_size) * 0.5 + rng.randn(grid_size) * 0.05
    v = np.ones(grid_size) * 0.25 + rng.randn(grid_size) * 0.05
    
    D_u, D_v = 1.0, 0.5
    kernel = np.array([1, -2, 1], dtype=float)
    
    for _ in range(200):
        lap_u = convolve1d(u, kernel, mode='wrap')
        lap_v = convolve1d(v, kernel, mode='wrap')
        uv2 = u * v * v
        u += 0.5 * (D_u * lap_u - uv2 + f * (1 - u))
        v += 0.5 * (D_v * lap_v + uv2 - (k + f) * v)
        u = np.clip(u, 0, 1)
        v = np.clip(v, 0, 1)
    
    # Threshold to pattern
    pattern = (u > np.median(u)).astype(float)
    
    # Velocity variation from v values
    velocities = 0.3 + 0.7 * v / (v.max() + 1e-8)
    
    # Synthesize
    total_samples = int(duration_sec * sr)
    result = np.zeros(total_samples, dtype=np.float32)
    samples_per_step = total_samples // grid_size
    
    for i in range(grid_size):
        if pattern[i] > 0.5:
            pos = i * samples_per_step
            # Simple hi-hat synthesis
            hat_dur = int(sr * 0.025)  # 25ms
            t = np.arange(hat_dur) / sr
            noise = np.random.randn(hat_dur) * 0.3
            env = np.exp(-t * 150)  # Fast decay
            hat = noise * env * velocities[i]
            # Bandpass 6-12kHz
            from scipy.signal import butter, lfilter
            b, a = butter(2, [6000, 12000], btype='band', fs=sr)
            hat = lfilter(b, a, hat)
            end = min(pos + len(hat), total_samples)
            result[pos:end] += hat[:end-pos]
    
    return result, pattern, velocities


# Generate patterns at different parameters
print('=== Turing Hi-Hat Patterns ===')
print('Reaction-diffusion in 1D time → organic percussion')
print()

tempo = 90  # Default
if beat_dimensions:
    tempo = beat_dimensions[0].get('tempo', 90)

fig, axes = plt.subplots(3, 1, figsize=(14, 6))
patterns_generated = []

for i, (name, f_val, k_val) in enumerate([
    ('Sparse (boom-bap)', 0.035, 0.065),
    ('Medium (neo-soul)', 0.055, 0.062),
    ('Dense (trap)', 0.065, 0.058),
]):
    audio, pattern, velocities = turing_hihat_pattern(22050, 4.0, tempo, f=f_val, k=k_val, seed=42+i)
    
    hits = int(np.sum(pattern))
    total = len(pattern)
    density = hits / total * 100
    print(f'  {name}: {hits}/{total} hits ({density:.0f}% density)')
    
    axes[i].stem(range(len(pattern)), pattern * velocities, linefmt='k-', markerfmt='ko', basefmt='k-')
    axes[i].set_title(f'{name} — f={f_val}, k={k_val} ({hits} hits)')
    axes[i].set_xlim(0, len(pattern))
    axes[i].set_ylim(0, 1.2)
    
    sf.write(str(AUDIO_DIR / f'turing_hats_{name.split("(")[1].strip(")")}.wav'), audio, 22050)
    patterns_generated.append({'name': name, 'f': f_val, 'k': k_val, 'hits': hits, 'total': total})

plt.tight_layout()
plt.savefig(str(AUDIO_DIR / 'turing_hihat_patterns.png'), dpi=150)
plt.show()
print(f'\nSaved audio + plot to {AUDIO_DIR}')

In [ ]:
#@title 5. Energy Mycelium — does the beat grow toward signal?

def energy_growth_analysis(y, sr, n_segments=32):
    """Analyze how a beat's energy grows over time.
    Mycelium growth = energy climbing toward peaks.
    Linear = boring. Fractal = interesting."""
    
    segment_len = len(y) // n_segments
    energies = []
    
    for i in range(n_segments):
        start = i * segment_len
        end = start + segment_len
        rms = np.sqrt(np.mean(y[start:end] ** 2))
        energies.append(rms)
    
    energies = np.array(energies)
    energies_norm = energies / (energies.max() + 1e-8)
    
    # Measure growth pattern
    # Linear growth = D near 1. Fractal growth = D > 1.
    diffs = np.abs(np.diff(energies_norm))
    
    # Box-counting on energy transitions
    scales = [1, 2, 4, 8]
    counts = []
    for s in scales:
        downsampled = energies_norm[::s]
        transitions = np.sum(np.abs(np.diff(downsampled)) > 0.05)
        counts.append(max(transitions, 1))
    
    if len(counts) >= 2:
        D = np.polyfit(np.log(scales), np.log(counts), 1)[0]
        D = abs(D)
    else:
        D = 0
    
    # Find peak location (should be ~75% for good arrangement)
    peak_pos = np.argmax(energies_norm) / n_segments
    
    return {
        'energy_curve': energies_norm.tolist(),
        'growth_dimension': round(D, 3),
        'peak_position': round(peak_pos, 3),
        'dynamic_range': round(float(energies.max() / (energies.min() + 1e-8)), 1),
    }


print('=== Energy Growth Analysis ===')
print('Peak at ~0.75 = good arrangement (climax at 3/4)')
print('High D = fractal energy growth (interesting)')
print('Low D = flat energy (boring loop)')
print()

growth_results = []
fig, axes = plt.subplots(min(len(beats_to_analyze), 5), 1, figsize=(14, 3*min(len(beats_to_analyze), 5)))
if not hasattr(axes, '__len__'):
    axes = [axes]

for i, beat_path in enumerate(beats_to_analyze[:5]):
    try:
        y, sr = librosa.load(str(beat_path), sr=22050, duration=60)
        result = energy_growth_analysis(y, sr)
        result['name'] = beat_path.name
        growth_results.append(result)
        
        growth_type = 'fractal' if result['growth_dimension'] > 0.5 else 'linear'
        peak_quality = 'good' if 0.6 < result['peak_position'] < 0.85 else 'off'
        print(f'  {beat_path.name}: D={result["growth_dimension"]:.3f} ({growth_type}), peak at {result["peak_position"]:.0%} ({peak_quality})')
        
        axes[i].plot(result['energy_curve'], 'o-', color='saddlebrown')
        axes[i].axvline(x=result['peak_position'] * len(result['energy_curve']), color='gold', linestyle='--', alpha=0.7)
        axes[i].set_title(f'{beat_path.name} — D={result["growth_dimension"]:.3f}')
        axes[i].set_ylim(0, 1.1)
    except Exception as e:
        print(f'  {beat_path.name}: error — {e}')

plt.tight_layout()
plt.savefig(str(AUDIO_DIR / 'energy_growth.png'), dpi=150)
plt.show()

if growth_results:
    (AUDIO_DIR / 'growth_analysis.json').write_text(json.dumps(growth_results, indent=2))
    print(f'\nSaved to {AUDIO_DIR}')

In [ ]:
#@title 6. Session Summary

results = {
    'session_date': time.strftime('%Y-%m-%d %H:%M'),
    'beats_analyzed': len(beat_dimensions) if 'beat_dimensions' in dir() else 0,
    'patterns_generated': len(patterns_generated) if 'patterns_generated' in dir() else 0,
    'growth_analyzed': len(growth_results) if 'growth_results' in dir() else 0,
    'files_saved': [str(f) for f in AUDIO_DIR.glob('*')],
    'duration_min': round((time.time() - SESSION_START) / 60, 1),
}

(AUDIO_DIR / 'audio_session.json').write_text(json.dumps(results, indent=2))

print('=== Audio Fractal Lab — Session Summary ===')
print(f'Duration: {results["duration_min"]} min')
print(f'Beats analyzed for rhythmic D: {results["beats_analyzed"]}')
print(f'Turing patterns generated: {results["patterns_generated"]}')
print(f'Energy growth curves: {results["growth_analyzed"]}')
print(f'Files on Drive: {len(results["files_saved"])}')
print(f'Time remaining: {time_left()/60:.0f} min')
print()
print('Pangea-Earth. Same fractal toolkit. Different coordinate.')
print('Guinea Pig Trench LLC')